# 🏛️ 03 - SÉNATEUR V3 : FEATURE EDITION (Le Technocrate)

Ce notebook implémente le troisième membre du "Grand Conseil" : **Divination V3**.

### 🎯 Stratégie :
- Utiliser la base robuste de V1 (les mêmes modèles).
- **Ajouter des features expertes** issues de l'analyse de corrélation (Pages/Pays, New User x Pages, etc.).
- Objectif : Capturer des signaux faibles que V1 et V2 ratent.

### 🔬 Nouvelles Features :
- `pages_relative_to_country` : L'utilisateur est-il plus actif que la moyenne de son pays ?
- `new_user_pages` : Interaction directe.
- `is_hyper_active` : Seuil critique d'activité (>12 pages).

In [ ]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

In [ ]:
train_df = pd.read_csv('conversion_data_train.csv')
test_df = pd.read_csv('conversion_data_test.csv')

def advanced_feature_engineering(df, train_stats=None):
    df_eng = df.copy()
    
    # Base V1
    df_eng['is_active'] = (df_eng['total_pages_visited'] > 2).astype(int)
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    
    # V3 Exclusif
    df_eng['new_user_pages'] = df_eng['new_user'] * df_eng['total_pages_visited']
    df_eng['is_hyper_active'] = (df_eng['total_pages_visited'] > 12).astype(int)
    
    # Target Encoding Like (calculé sur train uniquement)
    if train_stats is None:
        country_means = df_eng.groupby('country')['total_pages_visited'].mean()
        train_stats = {'country_means': country_means}
    
    means = train_stats['country_means']
    df_eng['country_mean_pages'] = df_eng['country'].map(means).fillna(means.mean())
    df_eng['pages_relative_to_country'] = df_eng['total_pages_visited'] / df_eng['country_mean_pages']
    
    return df_eng, train_stats

print("🔧 Création des features...")
X, train_stats = advanced_feature_engineering(train_df.drop('converted', axis=1))
y = train_df['converted']
X_test, _ = advanced_feature_engineering(test_df, train_stats)

print(f"✅ Features V3 prêtes : {list(X.columns)}")

In [ ]:
# Preprocessor V3 (inclut les nouvelles colonnes)
num_cols = ['age', 'total_pages_visited', 'interaction_age_pages', 'pages_per_age', 
            'new_user_pages', 'is_hyper_active', 'pages_relative_to_country']
cat_cols = ['country', 'source']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

# Voting V1 Standard (mais avec nouvelles données)
voting_clf = VotingClassifier(
    estimators=[
        ('xgb1', XGBClassifier(n_estimators=350, max_depth=4, learning_rate=0.05, subsample=0.9, random_state=42)),
        ('xgb2', XGBClassifier(n_estimators=350, max_depth=4, learning_rate=0.05, subsample=0.9, random_state=2025)),
        ('lgbm', LGBMClassifier(n_estimators=350, max_depth=4, learning_rate=0.05, subsample=0.9, verbose=-1, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=350, max_depth=4, learning_rate=0.05, subsample=0.9, random_state=42)),
        ('logreg', LogisticRegression(class_weight={0: 1, 1: 80}))
    ],
    voting='soft',
    weights=[0.7, 0.7, 1.2, 0.7, 0.5], # Poids V1 Standard
    n_jobs=-1
)

pipeline = Pipeline([('pre', preprocessor), ('clf', voting_clf)])

In [ ]:
# Validation (Attention: recalculer les stats country pour chaque fold serait mieux, 
# mais ici on utilise le pipeline global pour simplifier le notebook)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y))

print("🚀 Cross-Validation...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Pour être rigoureux sur le Data Leakage, on devrait refaire le FE ici.
    # Dans ce notebook simplifié, on assume que le FE global est acceptable pour une estimation.
    pipeline.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof_preds[val_idx] = pipeline.predict_proba(X.iloc[val_idx])[:, 1]
    print(f"   Fold {fold}/5 terminé.")

best_thr, best_f1 = 0.5, 0
for thr in np.linspace(0.3, 0.6, 100):
    score = f1_score(y, (oof_preds >= thr).astype(int))
    if score > best_f1: best_f1, best_thr = score, thr

print(f"✅ V3 F1-Score : {best_f1:.5f} (Seuil : {best_thr:.3f})")

In [ ]:
pipeline.fit(X, y)
test_preds = (pipeline.predict_proba(X_test)[:, 1] >= best_thr).astype(int)

pd.DataFrame({'converted': test_preds}).to_csv('divination_v3_features_predictions.csv', index=False)
print(f"💾 Prédictions V3 sauvegardées ({test_preds.sum()} conversions)")